In [49]:
import torch 
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [50]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [51]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor  = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor  = torch.from_numpy(y_test.astype(np.float32))

In [52]:
class LogisticReg(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

    def criterion(self, yhat, y):
        return nn.BCELoss()(yhat, y)

In [53]:
num_epoch = 1000

model = LogisticReg(X_train.shape[1], 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(num_epoch):
    model.train()
    optimizer.zero_grad()

    y_pred = model(X_train_tensor).squeeze()
    loss = model.criterion(y_pred, y_train_tensor)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.item():.4f}')

Epoch: 0, Loss: 0.6124
Epoch: 100, Loss: 0.2772
Epoch: 200, Loss: 0.2050
Epoch: 300, Loss: 0.1724
Epoch: 400, Loss: 0.1534
Epoch: 500, Loss: 0.1407
Epoch: 600, Loss: 0.1314
Epoch: 700, Loss: 0.1243
Epoch: 800, Loss: 0.1186
Epoch: 900, Loss: 0.1139


In [54]:

y_test_pred = model(X_test_tensor).squeeze()
y_test_cls  = (y_test_pred >= 0.5).float()
accuracy    = (y_test_cls == y_test_tensor).float().mean()
print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')

Test Accuracy: 99.12%
